# 🗂️ Python Hash Map / Hash Set — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> A hash map is a **coat check room**. You hand the attendant a key (your string/number), they give you a ticket number (the hash), and your item goes straight to that numbered hook. Retrieval is O(1) — walk to hook, grab coat. No scanning. No sorting. Just: key → hook → done.

---

## 📋 Table of Contents
| # | Section |
|---|--------|
| 1 | [What Is a Hash Map? The Visual Model](#1) |
| 2 | [Creating / Setup](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Store Complement — LC 1](#5) |
| 6 | [Pattern 2: Frequency Count — LC 347](#6) |
| 7 | [Pattern 3: Grouping by Key — LC 49](#7) |
| 8 | [Pattern 4: Sliding Window + Freq Map — LC 567](#8) |
| 9 | [Pattern 5: Two-Map Intersection — LC 242](#9) |
| 10 | [The Hash Map Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |

<a id='1'></a>

## 1. 💻 What Is a Hash Map? The Visual Model

```
HOW A HASH MAP WORKS
=====================

  key="cat"                key="dog"               key="bird"
      |                       |                        |
      v                       v                        v
  hash("cat")=3           hash("dog")=1           hash("bird")=5
      |                       |                        |
      v                       v                        v
 +---+---+---+---+---+---+---+---+
 | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |   <-- buckets (array of slots)
 +---+---+---+---+---+---+---+---+
       |       |       |   
     "dog"   "cat"   "bird"
      42      99       7       <-- stored values


DICT vs SET  (side by side)
============================

  dict {"cat": 99}           set {"cat", "dog", "bird"}
  +-----------+              +----------+
  | key | val |              |   key    |
  +-----+-----+              +----------+
  | cat |  99 |              |   cat    |
  | dog |  42 |              |   dog    |
  |bird |   7 |              |   bird   |
  +-----+-----+              +----------+
  Lookup: key -> value       Lookup: is key present?
  Use when: need to store    Use when: only need
  extra info per key         membership check


O(1) LOOKUP vs O(n) LIST SCAN
==============================

  List scan: is "bird" in ["cat","dog","fish","bird","ant"]?
  Step 1: check "cat"   -- no
  Step 2: check "dog"   -- no
  Step 3: check "fish"  -- no
  Step 4: check "bird"  -- YES  (4 steps, worst case n steps)

  Hash set: is "bird" in {"cat","dog","fish","bird","ant"}?
  Step 1: hash("bird") = 5, check bucket 5 -- YES  (1 step, always)

  With n=1,000,000 elements:
  List:  up to 1,000,000 comparisons
  Set:   still 1 step
```

**Why does this matter?**

The second your brain sees *"check if seen before"* or *"find a matching pair"* in a problem, that is the hash map signal. A nested loop is O(n²). One pass with a hash map is O(n). On n=10,000, that is 100,000,000 operations vs 10,000. Hash map wins every time.

<a id='2'></a>

## 2. 🔧 Creating / Setup

In [ ]:
from collections import defaultdict, Counter

# --- DICT CREATION FORMS ---

empty = {}                                   # empty dict — start fresh
print("empty dict:        ", empty)

from_tuples = dict([("a", 1), ("b", 2)])     # dict from list of (key, val) tuples
print("from tuples:       ", from_tuples)

comprehension = {c: ord(c) for c in "abc"}  # dict comprehension — key: value
print("comprehension:     ", comprehension)

freq = defaultdict(int)                      # missing key auto-initializes to 0
freq["x"] += 1                               # no KeyError — just works
freq["x"] += 1
freq["y"] += 1
print("defaultdict(int):  ", dict(freq))

groups = defaultdict(list)                   # missing key auto-initializes to []
groups["vowel"].append("a")                  # no KeyError — just works
groups["vowel"].append("e")
groups["consonant"].append("b")
print("defaultdict(list): ", dict(groups))

counter = Counter("aabbccca")               # count occurrences in one call
print("Counter:           ", counter)

# --- SET CREATION FORMS ---

from_list = set([1, 2, 2, 3, 3, 3])         # set from list — deduplicates
print("set from list:     ", from_list)

frozen = frozenset([1, 2, 3])               # immutable set — can be used as dict key
print("frozenset:         ", frozen)

<a id='3'></a>

## 3. ⚡ The Core API — All Operations

```
OPERATION              COMPLEXITY   WHAT IT DOES
──────────────────────────────────────────────────────
d[key]                 O(1) avg     Get value — raises KeyError if missing
d.get(key, default)    O(1) avg     Get value — returns default if missing
d[key] = val           O(1) avg     Set or update value
key in d               O(1) avg     Membership check — dict key or set element
del d[key]             O(1) avg     Delete key — raises KeyError if missing
d.pop(key, default)    O(1) avg     Delete and return — safe with default
d.setdefault(k, v)     O(1) avg     Insert if missing, return value
d.items()              O(1)         View of (key, val) pairs — lazy
d.keys()               O(1)         View of keys — lazy
d.values()             O(1)         View of values — lazy
len(d)                 O(1)         Number of key-value pairs
Counter(iterable)      O(n)         Frequency count in one call
Counter.most_common(k) O(n log k)   Top-k by count
──────────────────────────────────────────────────────
THINGS YOU DO NOT DO:
❌  d[key] when unsure key exists — use d.get(key, default) instead
❌  for k in d: del d[k]  — mutating while iterating raises RuntimeError
❌  Using a list as a dict key — lists are not hashable; use tuple
❌  Assuming dict order in Python < 3.7 — use OrderedDict if needed
```

In [ ]:
from collections import defaultdict, Counter

words = ["apple", "banana", "apple", "cherry", "banana", "apple"]

# --- d[key] = val  (set / update) ---
freq = {}
for w in words:
    freq[w] = freq.get(w, 0) + 1    # get with default=0 avoids KeyError
print("freq after get+update:", freq)

# --- d.get(key, default) vs d[key] ---
print("get existing key:    ", freq.get("apple", 0))   # 3
print("get missing key:     ", freq.get("grape", 0))   # 0 -- no KeyError

# --- key in d (membership) ---
print("'apple' in freq:     ", "apple" in freq)        # True  -- O(1)
print("'grape' in freq:     ", "grape" in freq)        # False -- O(1)

# --- d.setdefault(k, v) ---
groups = {}
for w in words:
    groups.setdefault(len(w), []).append(w)  # init list if key missing, then append
print("groups by length:    ", groups)

# --- defaultdict(int) -- cleaner freq count ---
freq2 = defaultdict(int)
for w in words:
    freq2[w] += 1                            # auto-init to 0 on first access
print("defaultdict freq:    ", dict(freq2))

# --- Counter -- one-liner freq count ---
c = Counter(words)
print("Counter:             ", c)
print("most_common(2):      ", c.most_common(2))       # top 2 by frequency

# --- d.pop(key, default) -- safe delete ---
removed = freq2.pop("apple", None)           # removes and returns -- None if missing
print("popped 'apple':      ", removed)
print("freq2 after pop:     ", dict(freq2))

# --- items(), keys(), values() ---
print("items:               ", list(freq2.items()))
print("keys:                ", list(freq2.keys()))
print("values:              ", list(freq2.values()))

<a id='4'></a>

## 4. 🧠 Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                  WHAT TO DO
───────────────────────────────────────────────────────
"two numbers that sum to target"       store complement: seen[target-num]
"find pairs / find what's missing"     complement map
"count occurrences of each"            Counter(iterable) or defaultdict(int)
"top-k most frequent"                  Counter + heapq.nlargest or bucket sort
"group strings by property"            defaultdict(list), sorted key
"check if two strings are anagrams"    Counter(s) == Counter(t)
"substring contains all of t"         two freq maps, sliding window
"O(1) lookup of seen values"           set (not dict -- no value needed)
"deduplicate preserving order"         seen = set(); [x for x if x not in seen]
```

<a id='5'></a>

## 5. 🧩 Pattern 1: Store Complement — LC 1

```
PROBLEM:
  Given nums and target, return indices of two numbers that add to target.

TRICK:
  For each num, its complement is (target - num).
  If complement is already in seen map -> we found our pair.
  Otherwise, store num -> its index for future lookups.

SLOW MOTION TRACE:  nums=[2,7,11,15]  target=9

  i=0  num=2   complement=9-2=7    7 in seen? NO   seen={2:0}
  i=1  num=7   complement=9-7=2    2 in seen? YES  -> return [seen[2], 1] = [0,1]
  (stopped early -- found on second element)

  Additional trace:  nums=[3,2,4]  target=6
  i=0  num=3   complement=6-3=3    3 in seen? NO   seen={3:0}
  i=1  num=2   complement=6-2=4    4 in seen? NO   seen={3:0, 2:1}
  i=2  num=4   complement=6-4=2    2 in seen? YES  -> return [seen[2], 2] = [1,2]

KEY INSIGHT:
  We never need to look backwards through the array -- the map IS the backwards lookup.

TIME:  O(n) -- single pass through nums
SPACE: O(n) -- seen map holds up to n entries
```

In [ ]:
from typing import List

def two_sum(nums: List[int], target: int) -> List[int]:
    """
    LC 1 -- Two Sum
    Approach: one-pass hash map, store complement.
    Time:  O(n) -- single pass through nums
    Space: O(n) -- seen map stores up to n entries
    """
    seen = {}                              # maps num -> index where we saw it

    for i, num in enumerate(nums):
        complement = target - num          # the number we NEED to have seen already

        if complement in seen:             # O(1) lookup -- complement was stored earlier
            print(f"  found: nums[{seen[complement]}]={complement} + nums[{i}]={num} = {target}")
            return [seen[complement], i]   # earlier index first

        seen[num] = i                      # not found yet -- store this num for future iterations
        print(f"  i={i} num={num} complement={complement} not in seen yet, seen={seen}")

    return []                              # problem guarantees a solution exists; this is a safety return


# slow motion trace visible in prints above

def test_harness(fn):
    cases = [
        # (args_tuple, expected)
        (([2, 7, 11, 15], 9),  [0, 1]),
        (([3, 2, 4],      6),  [1, 2]),
        (([3, 3],         6),  [0, 1]),
    ]
    for (nums, target), expected in cases:
        print(f"\nnums={nums} target={target}")
        result = fn(nums, target)
        status = "PASS" if result == expected else f"FAIL (got {result}, expected {expected})"
        print(f"  result={result} -> {status}")

test_harness(two_sum)
print("two_sum defined.")

<a id='6'></a>

## 6. 🧩 Pattern 2: Frequency Count — LC 347

```
PROBLEM:
  Given nums and k, return the k most frequent elements.

TRICK:
  Step 1: Count frequencies with Counter.
  Step 2: Use heapq.nlargest to pull top-k by frequency.
  Alternative: bucket sort -- index = frequency, bucket[i] = list of nums with freq i.
  heapq is O(n log k), bucket sort is O(n) but more code.

SLOW MOTION TRACE:  nums=[1,1,1,2,2,3]  k=2

  Step 1 -- count:
    Counter({1: 3, 2: 2, 3: 1})

  Step 2 -- heapq.nlargest(2, counts, key=counts.get):
    Compare counts: 1->3, 2->2, 3->1
    Top 2 by count: [1, 2]

  Result: [1, 2]

  Bucket sort alternative:
    bucket[3] = [1]     (1 appears 3 times)
    bucket[2] = [2]     (2 appears 2 times)
    bucket[1] = [3]     (3 appears 1 time)
    Read from right: [1, 2]  (first k=2 elements)

KEY INSIGHT:
  Counter does the O(n) count; nlargest handles the ranking.
  Never sort the full array for "top k" -- that is O(n log n); heap is O(n log k).

TIME:  O(n log k) -- Counter O(n) + nlargest O(n log k)
SPACE: O(n) -- counter stores up to n distinct elements
```

In [ ]:
import heapq
from collections import Counter
from typing import List

def top_k_frequent(nums: List[int], k: int) -> List[int]:
    """
    LC 347 -- Top K Frequent Elements
    Approach: Counter for frequencies, heapq.nlargest for top-k.
    Time:  O(n log k) -- Counter O(n) + nlargest O(n log k)
    Space: O(n) -- counter holds up to n distinct elements
    """
    # Step 1: count every element's frequency -- O(n)
    counts = Counter(nums)
    print(f"  counts: {counts}")

    # Step 2: pull top-k elements ranked by their count -- O(n log k)
    # key=counts.get means: use counts[element] as the sort key
    result = heapq.nlargest(k, counts, key=counts.get)
    print(f"  top {k} by frequency: {result}")

    return result


def top_k_frequent_bucket(nums: List[int], k: int) -> List[int]:
    """
    LC 347 -- Top K Frequent Elements (bucket sort variant)
    Approach: frequency buckets indexed by count -> read right to left.
    Time:  O(n) -- no sorting, just counting and reading buckets
    Space: O(n) -- buckets + counter
    """
    counts = Counter(nums)                  # num -> frequency

    # bucket[i] = list of numbers that appear exactly i times
    # max frequency is at most len(nums), so bucket size is len(nums)+1
    bucket = [[] for _ in range(len(nums) + 1)]
    for num, freq in counts.items():
        bucket[freq].append(num)            # place num in its frequency slot

    print(f"  bucket (non-empty): {[(i,b) for i,b in enumerate(bucket) if b]}")

    # collect results from highest frequency down
    result = []
    for freq in range(len(bucket) - 1, 0, -1):  # right to left
        for num in bucket[freq]:
            result.append(num)
            if len(result) == k:            # stop as soon as we have k
                return result

    return result


def test_harness(fn):
    cases = [
        (([1, 1, 1, 2, 2, 3], 2), {1, 2}),
        (([1],                1), {1}),
        (([4, 4, 4, 5, 5, 6], 2), {4, 5}),
    ]
    for (nums, k), expected in cases:
        print(f"\nnums={nums} k={k}")
        result = fn(nums, k)
        status = "PASS" if set(result) == expected else f"FAIL (got {result}, expected {expected})"
        print(f"  result={result} -> {status}")

print("=== heapq approach ===")
test_harness(top_k_frequent)
print("\n=== bucket sort approach ===")
test_harness(top_k_frequent_bucket)
print("top_k_frequent defined.")

<a id='7'></a>

## 7. 🧩 Pattern 3: Grouping by Key — LC 49

```
PROBLEM:
  Given a list of strings, group the anagrams together.

TRICK:
  Anagrams share the same sorted character sequence.
  sorted("eat") = ['a','e','t'] = tuple('aet') -> use as canonical dict key.
  defaultdict(list) accumulates all words that map to the same key.

SLOW MOTION TRACE:  strs=["eat","tea","tan","ate","nat","bat"]

  "eat" -> sorted -> ('a','e','t') -> groups[('a','e','t')] = ["eat"]
  "tea" -> sorted -> ('a','e','t') -> groups[('a','e','t')] = ["eat","tea"]
  "tan" -> sorted -> ('a','n','t') -> groups[('a','n','t')] = ["tan"]
  "ate" -> sorted -> ('a','e','t') -> groups[('a','e','t')] = ["eat","tea","ate"]
  "nat" -> sorted -> ('a','n','t') -> groups[('a','n','t')] = ["tan","nat"]
  "bat" -> sorted -> ('a','b','t') -> groups[('a','b','t')] = ["bat"]

  Result: [["eat","tea","ate"], ["tan","nat"], ["bat"]]

KEY INSIGHT:
  The canonical form is the fingerprint -- any two strings with the same
  fingerprint are anagrams. The map groups them automatically.

TIME:  O(n * m log m) -- n strings, each sorted in O(m log m) where m = avg length
SPACE: O(n * m) -- storing all strings in the groups map
```

In [ ]:
from collections import defaultdict
from typing import List

def group_anagrams(strs: List[str]) -> List[List[str]]:
    """
    LC 49 -- Group Anagrams
    Approach: sorted string as canonical key, defaultdict(list) accumulates groups.
    Time:  O(n * m log m) -- n strings each sorted in O(m log m)
    Space: O(n * m) -- all strings stored in groups map
    """
    groups = defaultdict(list)              # canonical_key -> [words with that fingerprint]

    for word in strs:
        key = tuple(sorted(word))           # sort chars -> make tuple (tuples are hashable, lists are not)
        groups[key].append(word)            # auto-init list on first access
        print(f"  '{word}' -> key={key} -> group now {list(groups[key])}")

    result = list(groups.values())
    print(f"  final groups: {result}")
    return result


def test_harness(fn):
    cases = [
        (
            (["eat", "tea", "tan", "ate", "nat", "bat"],),
            [["eat", "tea", "ate"], ["tan", "nat"], ["bat"]]
        ),
        (
            ([""],),
            [[""]]                           # single empty string is its own group
        ),
        (
            (["a"],),
            [["a"]]
        ),
    ]
    for args, expected in cases:
        print(f"\nstrs={args[0]}")
        result = fn(*args)
        # sort inner lists and outer list for order-independent comparison
        result_sorted = sorted([sorted(g) for g in result])
        expected_sorted = sorted([sorted(g) for g in expected])
        status = "PASS" if result_sorted == expected_sorted else f"FAIL (got {result_sorted}, expected {expected_sorted})"
        print(f"  result={result_sorted} -> {status}")

test_harness(group_anagrams)
print("group_anagrams defined.")

<a id='8'></a>

## 8. 🧩 Pattern 4: Sliding Window + Freq Map — LC 567

```
PROBLEM:
  Given strings s1 and s2, return True if any permutation of s1
  is a substring of s2.

TRICK:
  A permutation of s1 has the exact same character frequencies as s1.
  Slide a window of len(s1) across s2.
  Maintain a freq map of the window.
  If window_freq == s1_freq at any point -> found a permutation.
  Track number of matching char counts ('matches') to avoid full map comparison each step.

SLOW MOTION TRACE:  s1="ab"  s2="eidbaooo"

  s1_freq = {'a':1, 'b':1}   window_size = 2

  Build initial window s2[0:2] = "ei":
    window_freq = {'e':1, 'i':1}
    matches = 0  (no char in s1_freq matches window_freq counts)

  Slide: add s2[2]='d', remove s2[0]='e':
    window = "id"  window_freq = {'i':1, 'd':1}  matches = 0

  Slide: add s2[3]='b', remove s2[1]='i':
    window = "db"  window_freq = {'d':1, 'b':1}  matches = 1 ('b' matches)

  Slide: add s2[4]='a', remove s2[2]='d':
    window = "ba"  window_freq = {'b':1, 'a':1}  matches = 2 ('a' and 'b' match)
    matches == len(s1_freq) -> True!

KEY INSIGHT:
  Comparing full freq maps each step is O(26). Track a 'matches' integer instead.
  Increment matches when a char count aligns; decrement when it falls out of alignment.
  When matches == number of distinct chars in s1, the window IS a permutation.

TIME:  O(n) -- single pass through s2; each char added/removed once
SPACE: O(1) -- freq maps bounded by 26 lowercase letters
```

In [ ]:
from collections import Counter

def check_inclusion(s1: str, s2: str) -> bool:
    """
    LC 567 -- Permutation in String
    Approach: fixed-size sliding window with two freq maps and a 'matches' counter.
    Time:  O(n) -- single pass through s2
    Space: O(1) -- freq maps bounded by 26 lowercase letters
    """
    if len(s1) > len(s2):                   # s1 can't fit in s2
        return False

    s1_freq = Counter(s1)                   # frequency map of what we're looking for
    window_freq = Counter(s2[:len(s1)])     # frequency map of the current window

    # 'matches' = number of characters where window_freq[c] == s1_freq[c]
    # when matches == len(s1_freq) -> window is a permutation of s1
    matches = sum(1 for c in s1_freq if s1_freq[c] == window_freq[c])

    print(f"  s1_freq={dict(s1_freq)}  initial window='{s2[:len(s1)]}'  window_freq={dict(window_freq)}  matches={matches}")

    if matches == len(s1_freq):             # initial window is already a permutation
        return True

    for right in range(len(s1), len(s2)):  # slide window one char at a time
        add_char = s2[right]               # entering the window from the right
        remove_char = s2[right - len(s1)]  # leaving the window from the left

        # --- add new char on the right ---
        window_freq[add_char] += 1
        # check if adding this char caused a new match or broke one
        if add_char in s1_freq:
            if window_freq[add_char] == s1_freq[add_char]:
                matches += 1               # count just hit the target
            elif window_freq[add_char] == s1_freq[add_char] + 1:
                matches -= 1               # count just overshot the target

        # --- remove old char on the left ---
        window_freq[remove_char] -= 1
        if remove_char in s1_freq:
            if window_freq[remove_char] == s1_freq[remove_char]:
                matches += 1               # count fell back to exactly the target
            elif window_freq[remove_char] == s1_freq[remove_char] - 1:
                matches -= 1               # count fell below the target

        current_window = s2[right - len(s1) + 1: right + 1]
        print(f"  window='{current_window}'  matches={matches}")

        if matches == len(s1_freq):        # all char counts align -> permutation found
            return True

    return False


def test_harness(fn):
    cases = [
        (("ab",  "eidbaooo"), True),
        (("ab",  "eidboaoo"), False),
        (("adc", "dcda"),     True),
        (("a",   "ab"),       True),
    ]
    for (s1, s2), expected in cases:
        print(f"\ns1='{s1}'  s2='{s2}'")
        result = fn(s1, s2)
        status = "PASS" if result == expected else f"FAIL (got {result}, expected {expected})"
        print(f"  result={result} -> {status}")

test_harness(check_inclusion)
print("check_inclusion defined.")

<a id='9'></a>

## 9. 🧩 Pattern 5: Two-Map Intersection — LC 242

```
PROBLEM:
  Given two strings s and t, return True if t is an anagram of s.

TRICK:
  Two strings are anagrams if and only if they have identical character frequencies.
  Counter(s) == Counter(t) does this in one line.
  Alternative: freq array of 26 ints -- increment for s, decrement for t, all zeros means anagram.

SLOW MOTION TRACE:  s="anagram"  t="nagaram"

  Counter(s) = {'a':3, 'n':1, 'g':1, 'r':1, 'm':1}
  Counter(t) = {'n':1, 'a':3, 'g':1, 'r':1, 'm':1}
  Counter(s) == Counter(t) -> True

  Counter("rat") vs Counter("car"):
  {'r':1,'a':1,'t':1} vs {'c':1,'a':1,'r':1}
  't' != 'c' -> False

  Freq array approach on s="anagram" t="nagaram":
  Process s: a->+1, n->+1, a->+1, g->+1, r->+1, a->+1, m->+1
  Process t: n->-1, a->-1, g->-1, a->-1, r->-1, a->-1, m->-1
  Final array: all zeros -> anagram

KEY INSIGHT:
  Counter comparison is one-liner and O(n). Freq array avoids Counter overhead
  but both are O(n) time and O(1) space (bounded by 26 letters).

TIME:  O(n) -- Counter builds in O(n), comparison is O(1) (max 26 keys)
SPACE: O(1) -- bounded by 26 lowercase letters
```

In [ ]:
from collections import Counter

def is_anagram(s: str, t: str) -> bool:
    """
    LC 242 -- Valid Anagram  (Counter approach)
    Approach: build Counter for each string, compare equality.
    Time:  O(n) -- Counter builds in O(n), comparison O(1) (at most 26 keys)
    Space: O(1) -- bounded by 26 lowercase letters
    """
    if len(s) != len(t):                    # fast early exit -- different lengths can't be anagrams
        print(f"  lengths differ: {len(s)} vs {len(t)} -> False")
        return False

    s_count = Counter(s)
    t_count = Counter(t)
    print(f"  Counter(s)={dict(s_count)}")
    print(f"  Counter(t)={dict(t_count)}")

    result = s_count == t_count             # dict equality checks all key-value pairs
    print(f"  equal={result}")
    return result


def is_anagram_freq_array(s: str, t: str) -> bool:
    """
    LC 242 -- Valid Anagram  (freq array approach)
    Approach: single array of 26 ints; +1 for s chars, -1 for t chars.
    Time:  O(n) -- two passes through strings
    Space: O(1) -- fixed 26-element array
    """
    if len(s) != len(t):
        return False

    freq = [0] * 26                         # index 0='a', index 25='z'

    for c in s:
        freq[ord(c) - ord('a')] += 1        # increment count for each char in s

    for c in t:
        freq[ord(c) - ord('a')] -= 1        # decrement count for each char in t

    # if all zeros: every char in s was cancelled by the same char in t
    print(f"  freq array (non-zero only): {[(chr(i+ord('a')),v) for i,v in enumerate(freq) if v != 0]}")
    return all(v == 0 for v in freq)


def test_harness(fn):
    cases = [
        (("anagram", "nagaram"), True),
        (("rat",     "car"),     False),
        (("a",       "a"),       True),
        (("ab",      "a"),       False),
    ]
    for (s, t), expected in cases:
        print(f"\ns='{s}'  t='{t}'")
        result = fn(s, t)
        status = "PASS" if result == expected else f"FAIL (got {result}, expected {expected})"
        print(f"  result={result} -> {status}")

print("=== Counter approach ===")
test_harness(is_anagram)
print("\n=== Freq array approach ===")
test_harness(is_anagram_freq_array)
print("is_anagram defined.")

<a id='10'></a>

## 10. 🗺️ The Hash Map Decision Map

```
PATTERN                    WHEN YOU SEE THIS                    LC PROBLEMS
─────────────────────────────────────────────────────────────────────
Store Complement           "two numbers sum to target"          LC 1, LC 167
  seen[target - num]       "find pair with property"            LC 15 (variant)
  one-pass, O(n)           avoids nested loop O(n^2)

Frequency Count            "count occurrences"                  LC 347, LC 692
  Counter(iterable)        "top-k most frequent"                LC 451
  + heapq or bucket sort   "k closest / most/least common"

Grouping by Key            "group by property"                  LC 49, LC 249
  defaultdict(list)        "cluster strings"                    LC 819
  canonical key            sorted chars, freq tuple, etc.

Sliding Window + Freq      "permutation in string"              LC 567, LC 76
  two freq maps            "substring with all chars of t"      LC 438
  matches counter          "minimum window substring"

Two-Map Intersection       "are two strings anagrams?"          LC 242, LC 383
  Counter(s)==Counter(t)   "do frequencies match?"              LC 387
  or freq array            "same char counts?"
─────────────────────────────────────────────────────────────────────

QUICK SELECTOR:

  Need O(1) membership check only?         -> set()
  Need to store a value per key?           -> dict or defaultdict
  Need counts of elements?                 -> Counter
  Need to group items?                     -> defaultdict(list)
  Need top-k by frequency?                 -> Counter + heapq.nlargest
  Need fixed-size window with counts?      -> Counter + sliding window
  Need to compare char distributions?      -> Counter(s) == Counter(t)
```

<a id='11'></a>

## 11. 📝 Interview Cheat Sheet

---

### 1. When to Reach for HashMap

```
SIGNAL                                ACTION
───────────────────────────────────────────────────
"sum to target" / "find pair"         seen[target - x]
"count each element"                  Counter(arr) or defaultdict(int)
"top k most/least frequent"           Counter + heapq.nlargest(k, ...)
"group by some property"              defaultdict(list), canonical key
"anagram" / "same freq"               Counter(s) == Counter(t)
"permutation in substring"            sliding window + two freq maps
"seen before" / "visited"             set() -- no value needed
"O(1) lookup" explicitly asked        immediately think dict or set
```

---

### 2. O(1) Operations -- Memorize These

```python
# All of these are O(1) average:
val = d.get(key, 0)          # safe get with default
d[key] = val                 # insert or update
key in d                     # membership check
d.pop(key, None)             # safe delete
d.setdefault(key, [])        # init if missing
len(d)                       # size
elem in my_set               # set membership -- O(1), not O(n)
```

---

### 3. Common Templates

**Complement Map:**
```python
seen = {}
for i, num in enumerate(nums):
    if target - num in seen:
        return [seen[target - num], i]
    seen[num] = i
```

**Frequency Count:**
```python
from collections import Counter
counts = Counter(nums)              # or defaultdict(int)
top_k = heapq.nlargest(k, counts, key=counts.get)
```

**Group by Key:**
```python
from collections import defaultdict
groups = defaultdict(list)
for word in strs:
    groups[tuple(sorted(word))].append(word)
return list(groups.values())
```

**Sliding Window Freq:**
```python
s1_freq = Counter(s1)
window  = Counter(s2[:len(s1)])
matches = sum(1 for c in s1_freq if s1_freq[c] == window[c])
for r in range(len(s1), len(s2)):
    # add s2[r], remove s2[r - len(s1)]
    # update matches accordingly
    if matches == len(s1_freq): return True
```

**Two-Map Compare:**
```python
return Counter(s) == Counter(t)
```

---

### 4. Gotchas

```
WRONG                                    RIGHT
─────────────────────────────────────────────────────
❌ d[key] (key might be missing)         ✅ d.get(key, default)
❌ for k in d: del d[k]                  ✅ for k in list(d): del d[k]
❌ d[[1,2,3]] = val  (list not hashable)  ✅ d[(1,2,3)] = val  (tuple is hashable)
❌ sorted(d) for top-k                   ✅ heapq.nlargest(k, d, key=d.get)
❌ list scan for membership O(n)          ✅ set/dict membership O(1)
❌ Counter().most_common() -- full sort   ✅ Counter().most_common(k) -- heap O(n log k)
```

## Summary

```
Hash Map / Hash Set
|
+-- Core Data Structures
|   +-- dict {}               key -> value, O(1) get/set/delete
|   +-- set {}                key only, O(1) membership
|   +-- defaultdict(int)      auto-init to 0, freq counting
|   +-- defaultdict(list)     auto-init to [], grouping
|   +-- Counter               freq map + most_common
|
+-- Pattern 1: Store Complement
|   seen[target - num] -> O(n) from O(n^2)   [LC 1]
|
+-- Pattern 2: Frequency Count
|   Counter + heapq.nlargest                  [LC 347]
|   OR bucket sort for O(n)
|
+-- Pattern 3: Grouping by Key
|   canonical key (sorted chars)              [LC 49]
|   defaultdict(list) accumulates groups
|
+-- Pattern 4: Sliding Window + Freq Map
|   two Counters + 'matches' int              [LC 567]
|   fixed-size window slides across string
|
+-- Pattern 5: Two-Map Intersection
    Counter(s) == Counter(t)                  [LC 242]
    OR freq array [0]*26 for O(1) space
```

---

*End of Hash Map / Hash Set Master Guide — Sean Edition*